## step 1

In [ ]:
import pandas as pd
from glob import glob
import os

data_list = glob("/nas/houce/UK_mobility/processed_data/*_processed.csv")
data_list.sort(key=lambda x: os.path.basename(x).split('_')[1] + os.path.basename(x).split('_')[2])

In [ ]:
def process_and_save_locations(file_path, save_path):
    """
    Reads mobility data from a file, identifies home and workplace locations,
    and saves the results to specified directories.

    Args:
        file_path (str): The path to the input CSV file.
        save_path (str): The base directory where the output files will be saved.
    """
    try:
        df = pd.read_csv(file_path)
        df = df[df['showed_hours_today'] >= 12]
        if 'hours' in df.columns:
            df.drop(columns=['hours'], inplace=True)
            df = df.drop_duplicates()


        file_name_base = os.path.basename(file_path).replace('_processed.csv', '')

        # --- Identify Home Locations ---
        # Find the location with the maximum duration for each device
        home_df = df.groupby('device_aid')['duration_location_hour'].max().reset_index()
        
        # Merge to flag home locations in the main dataframe
        df_with_home = pd.merge(df, home_df, on=['device_aid', 'duration_location_hour'], how='left', indicator=True)
        df_with_home['home'] = (df_with_home['_merge'] == 'both').astype(int)
        df_with_home.drop(columns=['_merge'], inplace=True)

        # Save home locations data
        # home_dir = os.path.join(save_path, 'home')
        # os.makedirs(home_dir, exist_ok=True)
        # home_output_path = os.path.join(home_dir, f'home_{file_name_base}.csv')
        # df_with_home[df_with_home['home'] == 1].to_csv(home_output_path, index=False)
        # print(f"Home locations saved to {home_output_path}")

        # --- Identify Workplace Locations ---
        # Find max duration among non-home locations
        workplace_df = df_with_home[df_with_home['home'] == 0].groupby('device_aid')['duration_location_hour'].max().reset_index()

        # Merge to flag workplace locations
        df_final = pd.merge(df_with_home, workplace_df, on=['device_aid', 'duration_location_hour'], how='left', indicator=True)
        df_final['workplace'] = (df_final['_merge'] == 'both').astype(int)
        df_final.drop(columns=['_merge'], inplace=True)

        # Save workplace locations data
        # workplace_dir = os.path.join(save_path, 'workplace')
        # os.makedirs(workplace_dir, exist_ok=True)
        # workplace_output_path = os.path.join(workplace_dir, f'workplace_{file_name_base}.csv')
        # df_final[df_final['workplace'] == 1].to_csv(workplace_output_path, index=False)
        # print(f"Workplace locations saved to {workplace_output_path}")

        # Save the combined dataframe
        combined_output_path = os.path.join(save_path, f'{file_name_base}_home_workplace.csv')
        df_final.to_csv(combined_output_path, index=False)
        print(f"Combined data saved to {combined_output_path}")

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
    except Exception as e:
        print(f"An error occurred while processing {file_path}: {e}")

# Example of how to use the function for all files in data_list
# You can run this in a new cell
# save_path = "/nas/houce/UK_mobility/data_20251226/"

# from tqdm import tqdm
# for i, file_path in tqdm(enumerate(data_list[4:7]), total=len(data_list[4:7])):
#     process_and_save_locations(file_path, save_path)

In [ ]:
import multiprocessing
from functools import partial
from tqdm import tqdm

# 确定要使用的CPU核心数，可以根据你的机器配置调整
# os.cpu_count() 会返回系统中的CPU核心数
num_processes = multiprocessing.cpu_count()

# 要处理的文件列表切片
files_to_process = data_list[60:67]
save_path = "/nas/houce/UK_mobility/data_20251226_new/"

# 使用 functools.partial 创建一个新函数，该函数只需要 file_path 作为参数
# save_path 参数被预先填充
worker_func = partial(process_and_save_locations, save_path=save_path)

# 创建一个进程池
# 使用 with 语句可以确保进程池在使用后被正确关闭
with multiprocessing.Pool(processes=num_processes) as pool:
    # 使用 imap_unordered 来并行处理文件，并用 tqdm 显示进度
    # imap_unordered 会在任务完成时立即返回结果，这使得进度条可以实时更新
    for _ in tqdm(pool.imap_unordered(worker_func, files_to_process), total=len(files_to_process)):
        pass

print("All files have been processed.")